In [ ]:
import torch
import torch.nn as nn
from networks import FullyConnectedNN, NoisyFullyConnectedNN, FixNoiseFullyConnectedNN, DropoutFullyConnectedNN
import os
import pandas as pd
import numpy as np
from utils import *
import shutil

In [ ]:
dataset_name = 'bostonHousing'
X, y = load_uci_data_full(dataset_name)

In [ ]:
eval_path = os.path.join(os.getcwd(), 'evaluations')
n_folds = 5
inverted_cv_fraction = 5
# Create eval dir for dataset
dataset_path = os.path.join(eval_path, dataset_name)
# Generate folds for this dataset
folds_path = os.path.join(dataset_path, 'fold_indices')
batch_size = 32
optimizer_name = "Adam"
hidden_sizes = [50, 50]
learning_rate = 0.005
weight_decay = 0
use_cuda = torch.cuda.is_available()
epochs = 50
model_name = 'noisyFCN' # FCN, noisyFCN, fixnoiseFCN, dropoutFCN
init_noiselevel = 0.1
fix_noiselevel = 0.05
dropout_prob = 0.005  # 0.05, 0.005 according to MCDO

In [ ]:
# Get dataset configuration
feature_indices, target_indices = load_uci_info(dataset_name)
input_size = len(feature_indices)
output_size = len(target_indices)

In [ ]:
def def_optm(optimizer_name):
    global normal_param, learning_rate, momentum, weight_decay
    if optimizer_name == "SGD":
        print("using SGD as optimizer")
        optimizer = torch.optim.SGD([
                                    {'params': normal_param},
                                    {'params': alpha_param, 'weight_decay': 0}
                                    ],
                                    lr=learning_rate,
                                    momentum=momentum, weight_decay=weight_decay,
                                    nesterov=True)

    elif optimizer_name == "Adam":
        print("using Adam as optimizer")
        optimizer = torch.optim.Adam([
                                    {'params': normal_param},
                                    {'params': alpha_param, 'weight_decay': 0}
                                    ],
                                    lr=learning_rate,
                                    weight_decay=weight_decay)
    elif optimizer_name == "RMSprop":
        print("using RMSprop as optimizer")
        optimizer = torch.optim.RMSprop([
                                    {'params': normal_param},
                                    {'params': alpha_param, 'weight_decay': 0}
                                    ],
                                    lr=learning_rate, alpha=0.99, eps=1e-08, weight_decay=weight_decay, momentum=0)
    return optimizer

def def_model(model_name, input_size, hidden_sizes, output_size):
    if model_name == 'FCN':
        net = FullyConnectedNN(input_size=input_size, hidden_sizes=hidden_sizes, output_size=output_size)
    elif model_name == 'noisyFCN':
        net = NoisyFullyConnectedNN(input_size=input_size, hidden_sizes=hidden_sizes, output_size=output_size, init_noiselevel=init_noiselevel)
    elif model_name == 'fixnoiseFCN':
        net = FixNoiseFullyConnectedNN(input_size=input_size, hidden_sizes=hidden_sizes, output_size=output_size, fix_noiselevel=fix_noiselevel)
    elif model_name == 'dropoutFCN':
        net = DropoutFullyConnectedNN(input_size=input_size, hidden_sizes=hidden_sizes, output_size=output_size, dropout_prob=dropout_prob)
    
    return net

def train(model, train_loader, optimizer, criterion):
    model.train()
    losses = AverageMeter()
    for inputs, labels in train_loader:
        if use_cuda:
            labels = labels.cuda()
            inputs = inputs.cuda()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        losses.update(loss.item(), inputs.size(0))
    return losses.avg

def validate(model, val_loader, criterion):
    model.eval()
    if 'Dropout' in model._get_name():
        recursive_module_iteration(model)
    losses = AverageMeter()
    for inputs, labels in val_loader:
        if use_cuda:
            labels = labels.cuda()
            inputs = inputs.cuda()
        # compute output
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        losses.update(loss.item(), inputs.size(0))
    return losses.avg

def save_checkpoint(state, is_best, save_path, filename):
    filename = os.path.join(save_path, filename)
    torch.save(state, filename+'.pth.tar')
    if state['epoch'] % 10 == 0:
        torch.save(state, filename+'_epoch{}.pth.tar'.format(state['epoch']))
        last_filename = filename+'_epoch{}.pth.tar'.format(state['epoch']-10)
        if os.path.exists(last_filename):
            os.remove(last_filename)
    if is_best:  # copy the checkpoint to the best model if it is the best_loss
        bestname = os.path.join(save_path, 'model_best.pth.tar')
        shutil.copyfile(filename+'.pth.tar', bestname)
        print("=> Obtain best accuracy, and update the best model")

def recursive_module_iteration(module, depth=0):
    # 打印当前模块的信息
    print(f"{' ' * (depth * 2)}Module: {module.__class__.__name__}")

    # 如果当前模块是容器模块（如 Sequential），则递归遍历其子模块
    if hasattr(module, "children"):
        for child in module.children():
            recursive_module_iteration(child, depth + 1)
    if isinstance(module, nn.Dropout):
        module.train()
        print('dropout设置为训练模式')

In [ ]:
mean_val_loss = 0
for fold in range(n_folds):
    print('fold {}'.format(fold))
    X_train, y_train, X_val, y_val = load_fold(folds_path, fold, X, y)
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.float32)
    train_data = torch.utils.data.TensorDataset(X_train, y_train)
    val_data = torch.utils.data.TensorDataset(X_val, y_val)
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = torch.utils.data.DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=4)
    net = def_model(model_name, input_size, hidden_sizes, output_size)
    normal_param = [
        param for name, param in net.named_parameters()
        if (not 'alpha_' in name and not 'alphafix_' in name)
    ] # this is the parameters do not contain noise scale coefficient

    alpha_param = [
        param for name, param in net.named_parameters()
        if 'alpha_' in name
    ]
    criterion = nn.MSELoss()
    if use_cuda:
        net.cuda()
        criterion.cuda()
    optimizer = def_optm(optimizer_name)
    best_loss = float('inf')
    for epoch in range(epochs):
        train_loss = train(model=net, train_loader=train_loader, optimizer=optimizer, criterion=criterion)
        print('epoch {}'.format(epoch))
        print('train_loss:', train_loss)
        val_loss = validate(model=net, val_loader=val_loader, criterion=criterion)
        print('val_loss:', val_loss)

        checkpoint_state = {
            'epoch': epoch + 1,
            'model_name': model_name,
            'state_dict': net.state_dict(),
            'optimizer': optimizer.state_dict(),
        }

        is_best = False
        if val_loss < best_loss:
            is_best = True
            best_loss = val_loss

        if model_name == 'fixnoiseFCN':
            model_name = model_name + '_{}'.format(fix_noiselevel)
        if model_name == 'dropoutFCN':
            model_name = model_name + '_{}'.format(dropout_prob)
        save_path = os.path.join(dataset_path, 'save_model_cv', model_name+'_bs_{}_lr_{}_optm_{}'.format(batch_size, learning_rate, optimizer_name), 'fold_{}'.format(fold))
        if not os.path.exists(save_path):
            os.makedirs(save_path)
        save_checkpoint(checkpoint_state, is_best,
                        save_path, 'checkpoint')
    mean_val_loss += val_loss
mean_val_loss /= n_folds
print('mean_val_loss:', mean_val_loss)

In [ ]:
# 指定要写入的文件路径
file_path = 'results.txt'
res = 'datasets:{}, bs:{}, model:{}, optimizer:{}, lr:{}, weight_decay:{}, mean_val_loss:{}\n'.format(dataset_name, batch_size, net._get_name(), optimizer_name, learning_rate, weight_decay, mean_val_loss)
# 打开文件并写入内容
with open(file_path, 'a') as file:
    file.write(res)

In [ ]:
net._get_name()

In [ ]:
[(name, param) for name, param in net.named_parameters() if 'alpha' in name]